# JetRacer Final - Lane & Obstacle Avoidance

Sử dụng thuật toán nhận diện vạch đường mới nhất (1D Spatial Clustering) kết hợp né vật cản.

In [ ]:
import os
import time
import cv2
import numpy as np
import ipywidgets
import threading
import csv
from datetime import datetime
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from basic_motion import JetRacerController
import importlib
import lane_detection_v2
# Always reload the detector file so this notebook cannot retain an old
# lane-following implementation from a previous Jupyter run.
lane_detection_v2 = importlib.reload(lane_detection_v2)
get_detector = lane_detection_v2.get_detector
print('Lane detector loaded: curve preview + lane-safe obstacle avoidance')

# Restart NVArgus Daemon
os.system('echo "jetson" | sudo -S systemctl restart nvargus-daemon')
time.sleep(2)

try:
    if 'camera' in globals():
        camera.running = False
        camera.unobserve_all()
except:
    pass

camera = CSICamera(width=224, height=224, capture_fps=0)
car = JetRacerController()
detector = get_detector(224, 224)


In [ ]:
# Setup CSV Logging
log_filename = f"jetracer_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
with open(log_filename, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['timestamp', 'fps', 'detected_object', 'confidence', 'decision', 'latency_ms', 'control_output', 'event'])

# Setup UI Widgets
state_widget = ipywidgets.ToggleButtons(options=['stop', 'live'], description='State', value='stop')
raw_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
debug_widget = ipywidgets.Image(format='jpeg', width=224, height=224)

raw_widget.value = bgr8_to_jpeg(np.zeros((224, 224, 3), dtype=np.uint8))
debug_widget.value = bgr8_to_jpeg(np.zeros((224, 224, 3), dtype=np.uint8))

steering_gain_slider = ipywidgets.FloatSlider(description='Steering Gain', min=0.5, max=1.8, value=1.0, step=0.05)
throttle_slider = ipywidgets.FloatSlider(description='Max Throttle', min=0.06, max=0.25, value=0.12, step=0.01)

ui_widget = ipywidgets.VBox([
    ipywidgets.HBox([raw_widget, debug_widget]),
    ipywidgets.HBox([state_widget]),
    ipywidgets.HBox([steering_gain_slider, throttle_slider])
])

display(ui_widget)


In [ ]:
import time

frame_count = 0
last_time = time.time()
fps = 0.0
last_steering_command = 0.0
log_buffer = []

def live_update(change):
    global frame_count, last_time, fps, last_steering_command, log_buffer
    if state_widget.value != 'live':
        return
        
    start_time = time.time()
    img = change['new']
    
    # Process Frame
    debug_img, raw_steering, info = detector.process_frame(img, draw_debug=True)
    
    # Calculate Latency
    end_time = time.time()
    latency_ms = int((end_time - start_time) * 1000)
    
    # FPS Calculation
    frame_count += 1
    if end_time - last_time >= 1.0:
        fps = frame_count / (end_time - last_time)
        frame_count = 0
        last_time = end_time
        
    # Control Logic
    desired_steering = np.clip(raw_steering * steering_gain_slider.value, -1.0, 1.0)
    # Rate limiting suppresses one-frame glare/shadow spikes without hiding a real bend.
    # Limit sudden servo movement while retaining enough response for bends.
    max_steering_step = 0.12
    steering = float(np.clip(desired_steering,
                             last_steering_command - max_steering_step,
                             last_steering_command + max_steering_step))
    last_steering_command = steering
    # Slow down on sharp curves and obstacle manoeuvres; straight road keeps max speed.
    curve_scale = max(0.48, 1.0 - 0.52 * abs(steering))
    if info.get('lane_switching', False):
        curve_scale = min(curve_scale, 0.55)
    throttle = float(throttle_slider.value * curve_scale)
    
    # Fail safe: never continue with an old steering command after the
    # two lane boundaries have been lost for several consecutive frames.
    lane_confident = info.get('lane_confident', False)
    if lane_confident:
        car.set_steering(steering)
        car.set_throttle(throttle)
    else:
        steering = 0.0
        throttle = 0.0
        last_steering_command = 0.0
        car.stop()
    
    # Display Update
    raw_widget.value = bgr8_to_jpeg(img)
    debug_widget.value = bgr8_to_jpeg(debug_img)
    
    # Log to CSV
    obstacle = info.get('obstacle')
    detected_obj = 'Obstacle' if obstacle else 'Lane'
    decision = ('Lane Lost - Stop' if not lane_confident else
                ('Avoid Obstacle' if obstacle else 'Follow Lane'))
    control_out = f"S:{steering:.2f} T:{throttle:.2f}"
    event = info.get('case', '')
    
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
    log_buffer.append([timestamp, f"{fps:.1f}", detected_obj, int(lane_confident), decision, latency_ms, control_out, event])
    # Batch disk writes so logging does not destroy the >=20 FPS Speed Track score.
    if len(log_buffer) >= 20:
        with open(log_filename, mode='a', newline='') as f:
            csv.writer(f).writerows(log_buffer)
        log_buffer = []

def state_changed(change):
    global log_buffer
    if change['new'] == 'stop':
        car.stop()
        if log_buffer:
            with open(log_filename, mode='a', newline='') as f:
                csv.writer(f).writerows(log_buffer)
            log_buffer = []

state_widget.observe(state_changed, names='value')


In [ ]:
camera.observe(live_update, names='value')
camera.running = True
